<a href="https://colab.research.google.com/github/Sucheta-afk/Graphs-for-Emotion-Recognition/blob/main/draft_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall torch torch-geometric -y

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install torch-geometric

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 67.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 108.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 20.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 7.2 MB/s eta 0:00:00
    

In [2]:
import torch
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)
import os

2.5.1+cu121
2.7.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import output
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')


In [28]:
import os, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, numpy as np, pickle
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Load dataset ──────────────────────────────────────
save_dir = '/content/drive/MyDrive/SEED-IV/pyg_graphs/new_graph_files'
all_graphs = []
for fname in sorted([f for f in os.listdir(save_dir) if f.endswith('.pkl')]):
    with open(os.path.join(save_dir, fname), 'rb') as f:
        all_graphs.extend(pickle.load(f))
print(f"Loaded {len(all_graphs)} graphs.")

# save originals ONCE after loading
for g in all_graphs:
    g.x_original = g.x.clone()

# ── Hyperparameters ───────────────────────────────────
NUM_SUBJECTS    = 15
NUM_EMOTIONS    = 4
HIDDEN_CHANNELS = 128
GAT_HEADS       = 4
EPOCHS          = 30
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
BATCH_SIZE      = 64
LAMBDA_DOMAIN   = 0.5

# ── GRL ───────────────────────────────────────────────
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.save_for_backward(torch.tensor(alpha))
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        alpha = ctx.saved_tensors[0].item()
        return -alpha * grad_output, None

def grad_reverse(x, alpha=1.0):
    return GradientReversalFunction.apply(x, alpha)

# ── Model ─────────────────────────────────────────────
class EmotionGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels,
                 num_emotions, num_subjects, heads=4):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden_channels,
                             heads=heads, dropout=0.3)
        self.gat2 = GATConv(hidden_channels * heads, hidden_channels,
                             heads=1, concat=False, dropout=0.3)
        self.emotion_mlp = nn.Sequential(
            nn.Linear(hidden_channels, 64),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_emotions)
        )
        self.domain_mlp = nn.Sequential(
            nn.Linear(hidden_channels, 64),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_subjects)
        )

    def forward(self, x, edge_index, batch, alpha=1.0):
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        z = global_mean_pool(x, batch)
        emotion_out = self.emotion_mlp(z)
        domain_out  = self.domain_mlp(grad_reverse(z, alpha))
        return emotion_out, domain_out

# ── Helpers ───────────────────────────────────────────
def reset_features():
    for g in all_graphs:
        g.x = g.x_original.clone()

def normalize_features(source_graphs, target_graphs):
    all_x = torch.cat([g.x for g in source_graphs], dim=0)
    mean  = all_x.mean(dim=0)
    std   = all_x.std(dim=0) + 1e-8
    for g in source_graphs:
        g.x = (g.x - mean) / std
    for g in target_graphs:
        g.x = (g.x - mean) / std
    return source_graphs, target_graphs

def get_class_weights(source_graphs):
    labels = [g.y.item() for g in source_graphs]
    counts = Counter(labels)
    total  = len(labels)
    return torch.tensor(
        [total / (NUM_EMOTIONS * counts[i]) for i in range(NUM_EMOTIONS)],
        dtype=torch.float).to(DEVICE)

# ── Quick sanity check — 1 fold, 5 epochs, 50 graphs ──
def quick_check():
    source = [g for g in all_graphs if g.subject_id != 1][:200]
    target = [g for g in all_graphs if g.subject_id == 1][:50]

    source, target = normalize_features(source, target)
    class_weights  = get_class_weights(source)

    source_loader = DataLoader(source, batch_size=32, shuffle=True)
    target_loader = DataLoader(target, batch_size=32, shuffle=False)
    target_iter   = iter(target_loader)

    model     = EmotionGNN(5, HIDDEN_CHANNELS, NUM_EMOTIONS,
                           NUM_SUBJECTS, GAT_HEADS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, 6):  # just 5 epochs
        model.train()
        alpha = 2.0 / (1 + np.exp(-10 * epoch / 5)) - 1
        for src_batch in source_loader:
            src_batch = src_batch.to(DEVICE)
            tgt_batch = next(target_iter, None)
            if tgt_batch is None:
                target_iter = iter(target_loader)
                tgt_batch = next(target_iter)

            tgt_batch = tgt_batch.to(DEVICE)

            optimizer.zero_grad()
            emo_logits, dom_logits_src = model(
                src_batch.x, src_batch.edge_index, src_batch.batch, alpha)
            L_emotion = F.cross_entropy(emo_logits, src_batch.y,
                                        weight=class_weights)
            _, dom_logits_tgt = model(
                tgt_batch.x, tgt_batch.edge_index, tgt_batch.batch, alpha)
            src_dom = torch.zeros(src_batch.num_graphs, dtype=torch.long).to(DEVICE)
            tgt_dom = torch.ones(tgt_batch.num_graphs,  dtype=torch.long).to(DEVICE)
            L_domain = (F.cross_entropy(dom_logits_src, src_dom) +
                        F.cross_entropy(dom_logits_tgt, tgt_dom)) / 2
            loss = L_emotion - LAMBDA_DOMAIN * L_domain
            loss.backward()
            optimizer.step()

        # eval after each epoch
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in target_loader:
                batch = batch.to(DEVICE)
                out, _ = model(batch.x, batch.edge_index, batch.batch, alpha=0.0)
                preds.extend(out.argmax(dim=1).cpu().tolist())
                labels.extend(batch.y.cpu().tolist())
        acc = sum(p==l for p,l in zip(preds,labels)) / len(labels)
        print(f"Epoch {epoch}/5 — Acc: {acc:.4f}")

# quick_check()



Using device: cuda
Loaded 37575 graphs.


In [ ]:

# ── LOSO ─────────────────────────────────────────────
def run_loso(use_grl=True, lam=0.5, save_path=None):
    if os.path.exists(save_path):
        existing           = pd.read_csv(save_path)
        completed_subjects = set(existing['subject'].tolist())
        results            = existing.to_dict('records')
        print(f"Resuming — already done: {completed_subjects}")
    else:
        completed_subjects = set()
        results            = []

    for target_subj in range(1, NUM_SUBJECTS + 1):
        print(f"\n{'━'*52}")
        print(f"  FOLD {target_subj}/{NUM_SUBJECTS}  —  test subject = {target_subj}")
        print(f"{'━'*52}")

        if target_subj in completed_subjects:
            print(f"[SKIP] Subject {target_subj} already done")
            continue

        # reset features before every fold
        reset_features()

        source = [g for g in all_graphs if g.subject_id != target_subj]
        target = [g for g in all_graphs if g.subject_id == target_subj]

        source, target = normalize_features(source, target)
        class_weights  = get_class_weights(source)

        source_loader = DataLoader(source, batch_size=BATCH_SIZE,
                                   shuffle=True, num_workers=2, pin_memory=True)
        target_loader = DataLoader(target, batch_size=BATCH_SIZE, shuffle=False)
        target_iter   = iter(target_loader)

        model     = EmotionGNN(5, HIDDEN_CHANNELS, NUM_EMOTIONS,
                               NUM_SUBJECTS, GAT_HEADS).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(),
                                     lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                                     step_size=20, gamma=0.5)

        for epoch in tqdm(range(1, EPOCHS + 1), desc=f"  Subject {target_subj}"):
            model.train()
            alpha = 2.0 / (1 + np.exp(-10 * epoch / EPOCHS)) - 1

            for src_batch in source_loader:
                src_batch = src_batch.to(DEVICE)
                try:
                    tgt_batch = next(target_iter)
                except StopIteration:
                    target_iter = iter(target_loader)
                    tgt_batch   = next(target_iter)
                tgt_batch = tgt_batch.to(DEVICE)

                optimizer.zero_grad()

                emo_logits, dom_logits_src = model(
                    src_batch.x, src_batch.edge_index, src_batch.batch, alpha)
                L_emotion = F.cross_entropy(emo_logits, src_batch.y,
                                            weight=class_weights)

                src_dom = torch.zeros(src_batch.num_graphs,
                                      dtype=torch.long).to(DEVICE)
                L_dom_src = F.cross_entropy(dom_logits_src, src_dom)

                _, dom_logits_tgt = model(
                    tgt_batch.x, tgt_batch.edge_index, tgt_batch.batch, alpha)
                tgt_dom = torch.ones(tgt_batch.num_graphs,
                                     dtype=torch.long).to(DEVICE)
                L_dom_tgt = F.cross_entropy(dom_logits_tgt, tgt_dom)

                L_domain = (L_dom_src + L_dom_tgt) / 2
                loss     = L_emotion - lam * L_domain if use_grl else L_emotion

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            scheduler.step()

        # evaluate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in target_loader:
                batch = batch.to(DEVICE)
                emo_logits, _ = model(batch.x, batch.edge_index,
                                      batch.batch, alpha=0.0)
                all_preds.extend(emo_logits.argmax(dim=1).cpu().tolist())
                all_labels.extend(batch.y.cpu().tolist())

        acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
        f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)

        print(f"\n  Subject {target_subj:2d}  |  Acc: {acc:.4f}  |  Macro-F1: {f1:.4f}")
        print(classification_report(all_labels, all_preds,
              target_names=['Neutral','Sad','Fear','Happy'], zero_division=0))

        results.append({'subject': target_subj, 'accuracy': acc, 'macro_f1': f1})
        pd.DataFrame(results).to_csv(save_path, index=False)  # uses save_path param

    return pd.DataFrame(results)

# ── Run ───────────────────────────────────────────────
CKPT = "/content/drive/MyDrive/SEED-IV/training_checkpoints/"
os.makedirs(CKPT, exist_ok=True)

print("\n" + "═"*52)
print("  RUNNING: GAT + GRL")
print("═"*52)
df_grl = run_loso(use_grl=True, lam=0.5,
                  save_path=f"{CKPT}/grl_after_changing_lr_channels_100ep_37k.csv")

print("\n" + "═"*52)
print("  RUNNING: GAT without GRL (baseline)")
print("═"*52)
df_nogrl = run_loso(use_grl=False, lam=0.0,
                    save_path=f"{CKPT}/nogrl_after_changing_lr_channels_100ep_37k.csv")

# ── Final comparison ──────────────────────────────────
print("\n" + "═"*52)
print("  FINAL COMPARISON")
print("═"*52)
print(f"  GAT + GRL  → Acc: {df_grl['accuracy'].mean():.4f} ± {df_grl['accuracy'].std():.4f}  |  F1: {df_grl['macro_f1'].mean():.4f}")
print(f"  GAT no GRL → Acc: {df_nogrl['accuracy'].mean():.4f} ± {df_nogrl['accuracy'].std():.4f}  |  F1: {df_nogrl['macro_f1'].mean():.4f}")
print("═"*52)



════════════════════════════════════════════════════
  RUNNING: GAT + GRL
════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FOLD 1/15  —  test subject = 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Subject 1:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
EPOCHS = 50
df_grl_50 = run_loso(use_grl=True, lam=0.5,
                     save_path=f"{CKPT}/grl_results_50ep.csv")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FOLD 1/15  —  test subject = 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Subject 1:   0%|          | 0/50 [00:00<?, ?it/s]


  Subject  1  |  Acc: 0.3333  |  Macro-F1: 0.2349
              precision    recall  f1-score   support

     Neutral       0.29      0.94      0.45        18
         Sad       0.00      0.00      0.00        18
        Fear       0.46      0.33      0.39        18
       Happy       1.00      0.06      0.11        18

    accuracy                           0.33        72
   macro avg       0.44      0.33      0.23        72
weighted avg       0.44      0.33      0.23        72


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FOLD 2/15  —  test subject = 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Subject 2:   0%|          | 0/50 [00:00<?, ?it/s]


  Subject  2  |  Acc: 0.3056  |  Macro-F1: 0.2739
              precision    recall  f1-score   support

     Neutral       0.17      0.06      0.08        18
         Sad       0.75      0.17      0.27        18
        Fear       0.33      0.50      0.40        18
       Happy       0.26      0.50      0.34        18

    accuracy                           0.31        72
   macro avg       0.38      0.31      0.27        72
weighted avg       0.38      0.31      0.27        72


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FOLD 3/15  —  test subject = 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


...


  Subject 13  |  Acc: 0.5417  |  Macro-F1: 0.5085
              precision    recall  f1-score   support

     Neutral       0.48      0.83      0.61        18
         Sad       0.29      0.11      0.16        18
        Fear       0.54      0.72      0.62        18
       Happy       0.90      0.50      0.64        18

    accuracy                           0.54        72
   macro avg       0.55      0.54      0.51        72
weighted avg       0.55      0.54      0.51        72


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FOLD 14/15  —  test subject = 14
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Subject 14:   0%|          | 0/50 [00:00<?, ?it/s]


  Subject 14  |  Acc: 0.2917  |  Macro-F1: 0.2621
              precision    recall  f1-score   support

     Neutral       0.00      0.00      0.00        18
         Sad       0.75      0.17      0.27        18
        Fear       0.36      0.72      0.48        18
       Happy       0.31      0.28      0.29        18

    accuracy                           0.29        72
   macro avg       0.36      0.29      0.26        72
weighted avg       0.36      0.29      0.26        72


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FOLD 15/15  —  test subject = 15
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Subject 15:   0%|          | 0/50 [00:00<?, ?it/s]


  Subject 15  |  Acc: 0.1528  |  Macro-F1: 0.1456
              precision    recall  f1-score   support

     Neutral       0.22      0.11      0.15        18
         Sad       0.00      0.00      0.00        18
        Fear       0.15      0.22      0.18        18
       Happy       0.24      0.28      0.26        18

    accuracy                           0.15        72
   macro avg       0.15      0.15      0.15        72
weighted avg       0.15      0.15      0.15        72



In [ ]:
print(f"  GAT + GRL  → Acc: {df_grl['accuracy'].mean():.4f} ± {df_grl['accuracy'].std():.4f}  |  F1: {df_grl['macro_f1'].mean():.4f}")
print(f"  GAT no GRL → Acc: {df_nogrl['accuracy'].mean():.4f} ± {df_nogrl['accuracy'].std():.4f}  |  F1: {df_nogrl['macro_f1'].mean():.4f}")
print(f"  GAT + 50ep GRL → Acc: {df_grl_50['accuracy'].mean():.4f} ± {df_grl_50['accuracy'].std():.4f}  |  F1: {df_grl_50['macro_f1'].mean():.4f}")

  GAT + GRL  → Acc: 0.3083 ± 0.0732  |  F1: 0.2448
  GAT no GRL → Acc: 0.3176 ± 0.0478  |  F1: 0.2753
  GAT + 50ep GRL → Acc: 0.3037 ± 0.0853  |  F1: 0.2448


# THis is something else bro.. i'm checking if this graph atleast will imporve accuracy

In [4]:

TRIAL_LABELS = {
    '1': [1,2,3,0,2,0,0,1,0,1,2,1,1,1,2,3,2,2,3,3,0,3,0,3],
    '2': [2,1,3,0,0,2,0,2,3,3,2,3,2,0,1,1,2,1,0,3,0,1,3,1],
    '3': [1,2,2,1,3,3,3,1,1,2,1,0,2,3,3,0,2,3,0,0,2,0,1,0]
}

print('Number of trial labels:', len(TRIAL_LABELS['1']))

Number of trial labels: 24


In [5]:

raw_base = '/content/drive/MyDrive/SEED-IV/eeg_raw_data'
feat_base = '/content/drive/MyDrive/SEED-IV/eeg_feature_smooth'

for session in ['1', '2', '3']:
    raw_files = sorted(os.listdir(os.path.join(raw_base, session)))
    feat_files = sorted(os.listdir(os.path.join(feat_base, session)))
    print(f'Session {session}: {len(raw_files)} raw files, {len(feat_files)} feature files')

Session 1: 15 raw files, 15 feature files
Session 2: 15 raw files, 15 feature files
Session 3: 15 raw files, 15 feature files


In [6]:
import scipy.io as sio
def load_subject_session(subject_idx, session, raw_base, feat_base, threshold=0.4):
    """
    subject_idx : 0-based index into the sorted file list
    session     : '1', '2', or '3'
    returns     : list of PyG Data objects, one per trial
    """
    raw_path  = os.path.join(raw_base, session)
    feat_path = os.path.join(feat_base, session)

    raw_files  = sorted(os.listdir(raw_path))
    feat_files = sorted(os.listdir(feat_path))

    raw_mat  = sio.loadmat(os.path.join(raw_path,  raw_files[subject_idx]))
    feat_mat = sio.loadmat(os.path.join(feat_path, feat_files[subject_idx]))

    graphs = []
    subject_id = subject_idx + 1 # Define subject_id once per subject

    for trial_idx in range(24):
        trial_num = trial_idx + 1  # keys are 1-indexed
        label = TRIAL_LABELS[session][trial_idx] # Define label here, before its use

        # raw EEG for this trial
        raw_keys = [k for k in raw_mat.keys() if k.endswith(f'eeg{trial_num}') and not k.startswith('__')]
        raw_key = raw_keys[0]

        # DE features for this trial
        feat_keys = [k for k in feat_mat.keys() if k.endswith(f'LDS{trial_num}') and not k.startswith('__')]
        feat_key = feat_keys[0]

        raw_eeg    = raw_mat[raw_key]          # (62, N_samples)
        de_trial   = feat_mat[feat_key]        # (62, T, 5)

        # Compute PLV once per trial using raw_eeg
        plv_matrix = compute_averaged_plv(raw_eeg)
        edge_index, edge_weight = plv_to_edge_index(plv_matrix, threshold=threshold)

        for t in range(de_trial.shape[1]):  # 42 windows
            de_window = de_trial[:, t, :]   # (62, 5)
            graph = build_graph(de_window, label, edge_index, edge_weight,
                        subject_id=subject_id,
                        trial_id=trial_num,
                        window_id=t)
            graphs.append(graph)

    return graphs

In [7]:
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 14),
    'beta':  (14, 31),
    'gamma': (31, 50)
}
from scipy.signal import butter, filtfilt


def bandpass_filter(eeg, lowcut, highcut, fs=200, order=4):
    """
    eeg   : shape (n_channels, n_samples)
    returns filtered eeg of same shape
    """
    nyq = fs / 2
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    # apply filter to each channel independently
    filtered = filtfilt(b, a, eeg, axis=1)
    return filtered
from scipy.signal import hilbert

# HAD TO MODIFY FUNCTION TO REDUCE GRAPH GENERATION TIME (5HRS TO 1HR)
def compute_plv_matrix(filtered_eeg):
    """
    filtered_eeg : shape (n_channels, n_samples) — already band-pass filtered
    returns      : plv_matrix of shape (n_channels, n_channels)
    """
    n_channels = filtered_eeg.shape[0]

    # Step 1: extract instantaneous phase for every channel
    analytic = hilbert(filtered_eeg, axis=1)       # (62, N_samples) complex
    phase = np.angle(analytic)                      # (62, N_samples) real, in radians

    # vectorized — no for loop over pairs
    exp_phase = np.exp(1j * phase)  # (62, N_samples)
    plv_matrix = np.abs(exp_phase @ exp_phase.conj().T) / filtered_eeg.shape[1]

    np.fill_diagonal(plv_matrix, 1.0)  # a channel is perfectly locked with itself
    return plv_matrix


def compute_averaged_plv(raw_eeg, fs=200):
    """
    raw_eeg : shape (62, N_samples)
    returns : averaged PLV matrix of shape (62, 62)
    """
    plv_all_bands = []

    for band_name, (low, high) in BANDS.items():
        filtered = bandpass_filter(raw_eeg, low, high, fs=fs)
        plv = compute_plv_matrix(filtered)
        plv_all_bands.append(plv)

    # stack → (5, 62, 62), then average across bands → (62, 62)
    averaged_plv = np.mean(np.stack(plv_all_bands, axis=0), axis=0)
    return averaged_plv
def plv_to_edge_index(plv_matrix, threshold=0.5):
    """
    plv_matrix : (62, 62) numpy array
    threshold  : keep edges where PLV > threshold
    returns    : edge_index tensor of shape (2, E)
                 edge_weight tensor of shape (E,)
    """
    n = plv_matrix.shape[0]
    source_nodes = []
    target_nodes = []
    edge_weights = []

    for i in range(n):
        for j in range(i+1, n):   # upper triangle only
            if plv_matrix[i, j] > threshold:
                # add both directions (undirected graph)
                source_nodes.extend([i, j])
                target_nodes.extend([j, i])
                edge_weights.extend([plv_matrix[i, j],
                                     plv_matrix[i, j]])

    edge_index = torch.tensor([source_nodes, target_nodes],
                               dtype=torch.long)
    edge_weight = torch.tensor(edge_weights, dtype=torch.float)

    return edge_index, edge_weight
from torch_geometric.data import Data

def build_graph(de_features, label, edge_index, edge_weight, subject_id, trial_id, window_id):
    """
    de_features : (62, 5) — node features for a single window
    label       : int — emotion label (0/1/2/3)
    edge_index  : pre-computed edge_index tensor for the trial
    edge_weight : pre-computed edge_weight tensor for the trial
    subject_id  : int
    trial_id    : int
    window_id   : int
    returns     : PyG Data object
    """
    # node features
    x = torch.tensor(de_features, dtype=torch.float)   # (62, 5)

    # label
    y = torch.tensor([label], dtype=torch.long)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_weight,
        y=y,
        subject_id=subject_id,
        trial_id=trial_id,
        window_id=window_id
    )
    return data

In [8]:
import time
def run_full_dataset(raw_base, feat_base, save_dir, threshold=0.4):
    total_subjects = 15
    sessions = ['1', '2', '3']

    for session in sessions:
        for subject_idx in range(total_subjects):
            subject_id = subject_idx + 1
            filename = f'subject_{subject_id:02d}_session_{session}_new.pkl'
            save_path = os.path.join(save_dir, filename)

            # RESUME SUPPORT — skip if already saved
            if os.path.exists(save_path):
                print(f'[SKIP] {filename} already exists')
                continue

            print(f'[PROCESSING] Subject {subject_id}, Session {session}...')
            start = time.time()

            try:
                graphs = load_subject_session(subject_idx, session,
                                              raw_base, feat_base, threshold)
                with open(save_path, 'wb') as f:
                    pickle.dump(graphs, f)

                elapsed = time.time() - start
                print(f'[DONE] {filename} — {len(graphs)} graphs — {elapsed:.1f}s')

            except Exception as e:
                print(f'[ERROR] Subject {subject_id}, Session {session}: {e}')
                continue  # skip and move on, don't crash the whole loop

In [11]:
run_full_dataset(raw_base, feat_base, save_dir, threshold=0.4)

[SKIP] subject_01_session_1_new.pkl already exists
[SKIP] subject_02_session_1_new.pkl already exists
[SKIP] subject_03_session_1_new.pkl already exists
[SKIP] subject_04_session_1_new.pkl already exists
[SKIP] subject_05_session_1_new.pkl already exists
[SKIP] subject_06_session_1_new.pkl already exists
[SKIP] subject_07_session_1_new.pkl already exists
[SKIP] subject_08_session_1_new.pkl already exists
[SKIP] subject_09_session_1_new.pkl already exists
[SKIP] subject_10_session_1_new.pkl already exists
[SKIP] subject_11_session_1_new.pkl already exists
[SKIP] subject_12_session_1_new.pkl already exists
[SKIP] subject_13_session_1_new.pkl already exists
[SKIP] subject_14_session_1_new.pkl already exists
[SKIP] subject_15_session_1_new.pkl already exists
[SKIP] subject_01_session_2_new.pkl already exists
[SKIP] subject_02_session_2_new.pkl already exists
[SKIP] subject_03_session_2_new.pkl already exists
[SKIP] subject_04_session_2_new.pkl already exists
[SKIP] subject_05_session_2_new

In [12]:
save_dir = '/content/drive/MyDrive/SEED-IV/pyg_graphs/new_graph_files/'

all_graphs = []
for fname in sorted([f for f in os.listdir(save_dir) if f.endswith('.pkl')]):
    with open(os.path.join(save_dir, fname), 'rb') as f:
        all_graphs.extend(pickle.load(f))

print(f"Total graphs: {len(all_graphs)}")  # expect ~45k

Total graphs: 37575


In [23]:
import torch

# 1. Check if a GPU is available
print(f"Is GPU available? {torch.cuda.is_available()}")

# 2. Get the name of the GPU
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# 3. Check which device a specific tensor is on
x = torch.tensor([1.0, 2.0]).to("cuda" if torch.cuda.is_available() else "cpu")
print(f"Tensor is on: {x.device}")


Is GPU available? True
GPU Name: Tesla T4
Tensor is on: cuda:0


In [26]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4
